# 🧱 Lab: Refactor an Unseen Notebook into a Clean Package

**Module: The Python That Runs Your Pipeline** | **Type: Wall Lab**

---

## Learning Objectives

By the end of this lab, you will be able to:

1. **Identify** notebook smells (cell-order dependence, hidden globals, magic constants) in unfamiliar code
2. **Extract** tangled notebook logic into pure functions with type hints
3. **Organize** those functions into a package that imports cleanly from anywhere
4. **Choose** between a dict, a dataclass, and a function for each piece of the refactored code

## Concepts Covered

| Concept | Gloss | From |
|---------|-------|------|
| Notebook Smells | Notebook smells happen when hidden cell-order state, globals, and magic constants make code work only in one run, not when reused in a pipeline or module. | mini-extract-pure-function |
| Pure Functions | A pure function takes inputs only through its parameters and produces results only via its return value, printing and side effects are the caller's job. | mini-extract-pure-function |
| Type Hints | Type hints are optional labels on variables and functions that describe what kinds of values you expect, helping tools catch bugs and making code easier to read and refactor. | mini-add-type-hints |
| Modules & Packages | Files (modules) and folders with __init__.py (packages) make code importable from anywhere, so your pipeline logic is reusable, testable, and easier to run outside notebooks. | mini-build-triage-package |
| Dataclass as a Shape | Use a function for behavior and a dataclass for shape: a named, typed bundle of fields that makes pipeline inputs and outputs explicit. Use a class only when you truly need both state and behavior together. | mini-add-type-hints |

## Prerequisites

- **mini-extract-pure-function**: Spotting notebook smells and extracting inline logic into pure functions
- **mini-add-type-hints**: Type hints, and choosing between dicts, dataclasses, and full classes
- **mini-build-triage-package**: Creating modules, packages, and `__init__.py`


## The Scenario

A colleague left behind a notebook that processes support tickets — it classifies them by urgency, extracts key details, and produces a summary report. The notebook *works* if you run cells top-to-bottom, but it's riddled with notebook smells. Nobody dares touch it.

Your job: **audit it, refactor it, and ship it as a clean package** that a CLI script or web server could import.

We'll work through four phases:

1. **Read & Audit** — identify the smells
2. **Extract Pure Functions** — pull logic out with type hints
3. **Introduce Dataclasses** — replace shapeless dicts
4. **Build the Package** — organize into importable modules

## Phase 1: The Messy Notebook

Below is the original notebook code, consolidated into a single cell so you can see all the smells at once. **Read it carefully** — in the next step you'll list every smell you find.

In [ ]:
# ============================================================
# THE ORIGINAL "MESSY" NOTEBOOK  (read-only — do NOT fix here)
# ============================================================

# --- Cell 1: some setup ---
tickets = [
    {"id": "T-101", "text": "Server is DOWN. All API endpoints returning 503. Customers affected.", "source": "email"},
    {"id": "T-102", "text": "Could you update the logo on the About page when you get a chance?", "source": "slack"},
    {"id": "T-103", "text": "Payment processing failing for 30% of users since this morning.", "source": "email"},
    {"id": "T-104", "text": "Typo on FAQ page, third paragraph.", "source": "slack"},
    {"id": "T-105", "text": "Database connection pool exhausted. App restarting every 5 minutes.", "source": "pagerduty"},
]

# --- Cell 2: classify urgency ---
urgent_words = ["down", "failing", "exhausted", "critical", "outage", "crash"]
for t in tickets:
    words = t["text"].lower().split()
    hits = 0
    for w in words:
        for uw in urgent_words:
            if uw in w:
                hits += 1
    if hits >= 1:
        t["urgency"] = "high"
    else:
        t["urgency"] = "low"

# --- Cell 3: build report ---
high = []
low = []
for t in tickets:
    line = f"[{t['id']}] ({t['source']}) {t['text'][:60]}"
    if t["urgency"] == "high":
        high.append(line)
    else:
        low.append(line)

report = "URGENT TICKETS\n" + "=" * 40 + "\n"
for h in high:
    report += "  🔴 " + h + "\n"
report += f"\nLOW PRIORITY ({len(low)} tickets)\n" + "-" * 40 + "\n"
for l in low:
    report += "  ⚪ " + l + "\n"
report += f"\nTotal: {len(tickets)} | High: {len(high)} | Low: {len(low)}"

print(report)

## Phase 1b: Audit — Spot the Smells

Now let's catalog every smell. Recall the three categories from the audit mini-lab:

- **Cell-order dependence** — later cells silently depend on state created earlier
- **Hidden globals** — variables used across cells without being passed as arguments
- **Magic constants** — hard-coded values buried in logic

In [ ]:
from IPython.display import display, Markdown

def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))

audit = """
### 🔍 Smell Audit Results

| # | Smell | Type | Location |
|---|-------|------|----------|
| 1 | `tickets` list is a global that Cell 2 and Cell 3 mutate and read | Hidden global | Cells 1→2→3 |
| 2 | Cell 2 **mutates** each ticket dict in-place (`t["urgency"] = ...`) — Cell 3 only works if Cell 2 ran first | Cell-order dependence | Cell 2→3 |
| 3 | `urgent_words` list is hard-coded inside the classification logic | Magic constant | Cell 2 |
| 4 | Truncation length `60` is buried in the report builder | Magic constant | Cell 3 |
| 5 | `high` and `low` are globals used across the report builder — not returned from a function | Hidden global | Cell 3 |
| 6 | Everything is in dicts — `t["urgency"]`, `t["source"]` — no named shape, typos go unnoticed | Shapeless data | All cells |

**Verdict:** This notebook will break if cells are run out of order, and the logic can't be imported or tested from anywhere else.
"""

md(audit)

## Phase 2: Extract Pure Functions with Type Hints

We'll extract two pure functions from the messy code:

1. **`classify_urgency`** — takes a ticket text and a keyword list, returns `"high"` or `"low"`
2. **`build_report`** — takes a list of classified tickets, returns the report string

Each function will:
- Take **all inputs as parameters** (no globals)
- **Return** a result (no in-place mutation)
- Have **type hints** on every parameter and the return value

### Step 2a: Introduce a Dataclass for the Ticket Shape

Before writing the functions, let's replace the shapeless dicts with a proper dataclass. Recall from the dataclass mini-lab: if a bundle of values is passed between functions, it deserves a name and typed fields.

In [ ]:
from dataclasses import dataclass


@dataclass
class Ticket:
    """A support ticket with classification."""
    id: str
    text: str
    source: str
    urgency: str = "unclassified"   # default — gets set by classify_urgency


# Convert our raw dicts into Ticket instances
raw_tickets = [
    {"id": "T-101", "text": "Server is DOWN. All API endpoints returning 503. Customers affected.", "source": "email"},
    {"id": "T-102", "text": "Could you update the logo on the About page when you get a chance?", "source": "slack"},
    {"id": "T-103", "text": "Payment processing failing for 30% of users since this morning.", "source": "email"},
    {"id": "T-104", "text": "Typo on FAQ page, third paragraph.", "source": "slack"},
    {"id": "T-105", "text": "Database connection pool exhausted. App restarting every 5 minutes.", "source": "pagerduty"},
]

tickets = [Ticket(**t) for t in raw_tickets]
print(tickets[0])
print(f"\nField access: tickets[0].source → '{tickets[0].source}'")
print(f"IDE would catch: tickets[0].sorce  ← AttributeError, not silent None")

### Step 2b: `classify_urgency` — a Pure Function

The original code mutated each dict in-place. Our version takes a text string and returns a label — no side effects.

In [ ]:
DEFAULT_URGENT_KEYWORDS: list[str] = [
    "down", "failing", "exhausted", "critical", "outage", "crash"
]


def classify_urgency(
    text: str,
    urgent_keywords: list[str] = DEFAULT_URGENT_KEYWORDS,
) -> str:
    """Return 'high' if any urgent keyword appears in text, else 'low'."""
    text_lower = text.lower()
    for keyword in urgent_keywords:
        if keyword in text_lower:
            return "high"
    return "low"


# Test the pure function in isolation
print(classify_urgency("Server is DOWN"))          # → high
print(classify_urgency("Update the logo"))          # → low
print(classify_urgency("All good", ["bad", "fail"]))  # → low (custom keywords)

### Step 2c: Classify All Tickets

Instead of mutating dicts in-place, we create **new** `Ticket` instances with the urgency set. This avoids cell-order dependence — the original tickets are never modified.

In [ ]:
def classify_tickets(
    tickets: list[Ticket],
    urgent_keywords: list[str] = DEFAULT_URGENT_KEYWORDS,
) -> list[Ticket]:
    """Return new Ticket instances with urgency classified."""
    return [
        Ticket(
            id=t.id,
            text=t.text,
            source=t.source,
            urgency=classify_urgency(t.text, urgent_keywords),
        )
        for t in tickets
    ]


classified = classify_tickets(tickets)
for t in classified:
    print(f"  {t.id}: {t.urgency}")

### Step 2d: `build_report` — Another Pure Function

The original code used globals `high` and `low` and built the report by string concatenation with a magic constant `60` for truncation. Our version takes everything as parameters.

In [ ]:
def build_report(tickets: list[Ticket], max_text_length: int = 60) -> str:
    """Build a formatted urgency report from classified tickets."""
    high = [t for t in tickets if t.urgency == "high"]
    low = [t for t in tickets if t.urgency != "high"]

    lines: list[str] = []
    lines.append("URGENT TICKETS")
    lines.append("=" * 40)
    for t in high:
        truncated = t.text[:max_text_length]
        suffix = "..." if len(t.text) > max_text_length else ""
        lines.append(f"  🔴 [{t.id}] ({t.source}) {truncated}{suffix}")

    lines.append(f"\nLOW PRIORITY ({len(low)} tickets)")
    lines.append("-" * 40)
    for t in low:
        truncated = t.text[:max_text_length]
        suffix = "..." if len(t.text) > max_text_length else ""
        lines.append(f"  ⚪ [{t.id}] ({t.source}) {truncated}{suffix}")

    lines.append(f"\nTotal: {len(tickets)} | High: {len(high)} | Low: {len(low)}")
    return "\n".join(lines)


report = build_report(classified)
print(report)

## Phase 3: Before vs After Comparison

Let's compare what we've fixed.

In [ ]:
comparison = """
### Before → After

| Smell | Before | After |
|-------|--------|-------|
| Hidden globals | `tickets`, `high`, `low` used across cells | Every function takes inputs as parameters |
| Cell-order dependence | Cell 3 crashes if Cell 2 didn't run | `classify_tickets` → `build_report` — explicit data flow |
| Magic constants | `urgent_words` and `60` hard-coded | `DEFAULT_URGENT_KEYWORDS` named constant; `max_text_length` parameter |
| In-place mutation | `t["urgency"] = ...` mutates dicts | `classify_tickets` returns **new** `Ticket` instances |
| Shapeless dicts | `t["urgency"]` — typos invisible | `Ticket` dataclass — IDE catches `t.urgnecy` immediately |
| No type hints | Parameter types only known by reading body | Every function fully annotated |
"""

md(comparison)

## Phase 4: Build the Package

Now let's move everything out of the notebook and into an importable package called **`ticket_triage`**:

```
ticket_triage/
├── __init__.py        ← re-exports public API
├── models.py          ← Ticket dataclass
├── classifier.py      ← classify_urgency, classify_tickets
└── report.py          ← build_report
```

One job per module — exactly the pattern from the triage-package mini-lab.

In [ ]:
from pathlib import Path

pkg = Path("ticket_triage")
pkg.mkdir(exist_ok=True)
print(f"Package folder: {pkg.resolve()}")

In [ ]:
%%writefile ticket_triage/models.py
"""Data shapes for the ticket triage system."""

from dataclasses import dataclass


@dataclass
class Ticket:
    """A support ticket with classification."""
    id: str
    text: str
    source: str
    urgency: str = "unclassified"

In [ ]:
%%writefile ticket_triage/classifier.py
"""Ticket urgency classification."""

from .models import Ticket

DEFAULT_URGENT_KEYWORDS: list[str] = [
    "down", "failing", "exhausted", "critical", "outage", "crash"
]


def classify_urgency(
    text: str,
    urgent_keywords: list[str] = DEFAULT_URGENT_KEYWORDS,
) -> str:
    """Return 'high' if any urgent keyword appears in text, else 'low'."""
    text_lower = text.lower()
    for keyword in urgent_keywords:
        if keyword in text_lower:
            return "high"
    return "low"


def classify_tickets(
    tickets: list[Ticket],
    urgent_keywords: list[str] = DEFAULT_URGENT_KEYWORDS,
) -> list[Ticket]:
    """Return new Ticket instances with urgency classified."""
    return [
        Ticket(
            id=t.id,
            text=t.text,
            source=t.source,
            urgency=classify_urgency(t.text, urgent_keywords),
        )
        for t in tickets
    ]

In [ ]:
%%writefile ticket_triage/report.py
"""Report generation for classified tickets."""

from .models import Ticket


def build_report(tickets: list[Ticket], max_text_length: int = 60) -> str:
    """Build a formatted urgency report from classified tickets."""
    high = [t for t in tickets if t.urgency == "high"]
    low = [t for t in tickets if t.urgency != "high"]

    lines: list[str] = []
    lines.append("URGENT TICKETS")
    lines.append("=" * 40)
    for t in high:
        truncated = t.text[:max_text_length]
        suffix = "..." if len(t.text) > max_text_length else ""
        lines.append(f"  🔴 [{t.id}] ({t.source}) {truncated}{suffix}")

    lines.append(f"\nLOW PRIORITY ({len(low)} tickets)")
    lines.append("-" * 40)
    for t in low:
        truncated = t.text[:max_text_length]
        suffix = "..." if len(t.text) > max_text_length else ""
        lines.append(f"  ⚪ [{t.id}] ({t.source}) {truncated}{suffix}")

    lines.append(f"\nTotal: {len(tickets)} | High: {len(high)} | Low: {len(low)}")
    return "\n".join(lines)

In [ ]:
%%writefile ticket_triage/__init__.py
"""ticket_triage — classify support tickets and generate urgency reports."""

from .models import Ticket
from .classifier import classify_urgency, classify_tickets, DEFAULT_URGENT_KEYWORDS
from .report import build_report

__all__ = [
    "Ticket",
    "classify_urgency",
    "classify_tickets",
    "DEFAULT_URGENT_KEYWORDS",
    "build_report",
]

In [ ]:
# Verify file structure
for f in sorted(pkg.rglob("*.py")):
    print(f"  {f}")

## Phase 5: Use the Package

Now we import from `ticket_triage` exactly like any installed library — no `sys.path` hacks, no copy-pasting functions. This is the payoff of the refactor.

In [ ]:
from ticket_triage import Ticket, classify_tickets, build_report

# Fresh data — the package doesn't depend on any notebook state
new_tickets = [
    Ticket(id="T-201", text="Login service crash affecting all regions.", source="pagerduty"),
    Ticket(id="T-202", text="Can we add dark mode to the dashboard?", source="slack"),
    Ticket(id="T-203", text="CDN outage causing slow page loads worldwide.", source="email"),
    Ticket(id="T-204", text="Quarterly report formatting looks off.", source="email"),
]

classified = classify_tickets(new_tickets)
report = build_report(classified)
print(report)

## Cleanup

In [ ]:
import shutil

shutil.rmtree("ticket_triage", ignore_errors=True)
print("Cleaned up ticket_triage/ folder ✓")

## 🎯 Summary

### What You Built

You took a messy, smell-ridden notebook that classified support tickets and refactored it into a clean, importable `ticket_triage` package with typed dataclasses, pure functions, and separate modules — ready to be imported by any script, server, or test suite.

### Key Takeaways

1. **Smell audit first** — before writing any new code, catalog the globals, cell-order dependencies, and magic constants so you know exactly what to fix
2. **Pure functions eliminate cell-order bugs** — when every function takes inputs as parameters and returns a result, run order no longer matters
3. **Dataclasses replace shapeless dicts** — a `Ticket` dataclass catches typos at edit-time and makes the data contract explicit to every caller
4. **Type hints document the contract** — annotated signatures tell callers exactly what to pass and what they'll get back, without reading the function body
5. **One job per module** — `models.py`, `classifier.py`, `report.py` each own a single responsibility, and `__init__.py` re-exports the public API for clean imports